# Independent read-only audit of the merged dam dataset

This notebook receives only a completed run directory. It never imports or invokes the extraction runner. SQLite is opened read-only; scientific databases and CSVs are never written. Audit reports are the only outputs created.

## Audit boundary

The audit reconciles source identity, frozen engines, observation keys, outer-merge coverage, preserved rows/nulls, extraction quality evidence, SQLite integrity, CSV equivalence, investigation products, and before/after scientific artifact hashes. Status values are PASS, WARN, or FAIL.

In [1]:
from pathlib import Path
import sys, json, hashlib, sqlite3
import pandas as pd
from IPython.display import display
PROJECT_ROOT=Path.cwd().resolve()
if PROJECT_ROOT.name=="notebooks": PROJECT_ROOT=PROJECT_ROOT.parent
elif (PROJECT_ROOT/"DAM_DATA_PIPELINE_20260831").is_dir(): PROJECT_ROOT=PROJECT_ROOT/"DAM_DATA_PIPELINE_20260831"
RUN_DIR=PROJECT_ROOT/"outputs"/"runs"/"RUN_20260831_FULL_01"
AUDIT_DIR=RUN_DIR/"audit"
assert (RUN_DIR/"pipeline_receipt.json").is_file()
assert sys.version_info[:3]==(3,12,13)
print(sys.version); print(sys.executable); print("AUDIT_RUN_DIR =",RUN_DIR)

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:16:45) [MSC v.1942 64 bit (AMD64)]
<CONDA_ENV>\python.exe
AUDIT_RUN_DIR = <PFE_READY>\DAM_DATA_PIPELINE_20260831\outputs\runs\RUN_20260831_FULL_01


In [2]:
def sha256_file(path):
    d=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): d.update(b)
    return d.hexdigest()
scientific_paths=sorted([p for folder in ("page1","page2","database","exports","investigation","figures","reports") for p in (RUN_DIR/folder).rglob("*") if p.is_file()]+[RUN_DIR/"pipeline_receipt.json"])
before={str(p.relative_to(RUN_DIR)):sha256_file(p) for p in scientific_paths}
print("SCIENTIFIC_ARTIFACTS_HASHED_BEFORE =",len(before))

SCIENTIFIC_ARTIFACTS_HASHED_BEFORE = 55


In [3]:
receipt=json.loads((RUN_DIR/"pipeline_receipt.json").read_text(encoding="utf-8"))
manifest=pd.read_csv(RUN_DIR/"reports"/"source_manifest_sha256.csv",low_memory=False)
accepted=pd.read_csv(PROJECT_ROOT/"references"/"COMBINED_SOURCE_INVENTORY.csv",dtype=str)
p1=pd.read_csv(RUN_DIR/"page1"/"observations.csv.gz",low_memory=False)
p2=pd.read_csv(RUN_DIR/"page2"/"observations.csv.gz",low_memory=False)
merged=pd.read_csv(RUN_DIR/"exports"/"merged_dam_daily.csv",low_memory=False)
compressed=pd.read_csv(RUN_DIR/"exports"/"merged_dam_daily.csv.gz",low_memory=False)
p2n=p2.rename(columns={"dam_id":"canonical_dam_id"})
results=[]
def check(number,name,condition,details="",warn=False):
    status="PASS" if bool(condition) else ("WARN" if warn else "FAIL")
    results.append({"check_number":number,"check_name":name,"status":status,"details":str(details)})
roots={page:Path(group.source_path.iloc[0]).parent for page,group in manifest.groupby("page")}
check(1,"Source root identity",str(roots["page1"]).endswith(r"dam_pdfs_split\raw_data\1") and str(roots["page2"]).endswith(r"dam_pdfs_split\raw_data\2"),roots)
counts=manifest.groupby("page").size().to_dict(); check(2,"PDF counts",counts=={"page1":3981,"page2":3983},counts)
purity=manifest.groupby("page").apply(lambda g:g.source_file.str.lower().str.endswith("_1.pdf" if g.name=="page1" else "_2.pdf").all(),include_groups=False).all(); check(3,"Filename suffix purity",purity)
check(4,"Source manifest uniqueness",not manifest.duplicated(["page","source_file"]).any())
accepted_cmp=manifest.merge(accepted[["page","source_file","source_sha256"]],on=["page","source_file"],suffixes=("_actual","_accepted"),validate="one_to_one")
check(5,"Source hashes versus accepted inventory",len(accepted_cmp)==7964 and (accepted_cmp.source_sha256_actual==accepted_cmp.source_sha256_accepted).all())
check(6,"Page-1 engine version",receipt["page1_engine"]=="page1-authoritative-0.2.3",receipt["page1_engine"])
check(7,"Page-2 engine version",receipt["page2_engine"]=="page2-authoritative-1.0.0",receipt["page2_engine"])
frozen=pd.read_csv(PROJECT_ROOT/"investigation"/"ACCEPTED_SOURCE_HASHES.csv"); check(8,"Frozen source hashes",all(sha256_file(PROJECT_ROOT/r.destination_path)==r.sha256 for r in frozen.itertuples()))
check(9,"Page-1 source count",receipt["page1_summary"]["processed_source_count"]==3981)
check(10,"Page-2 source count",receipt["page2_summary"]["processed_source_count"]==3983)
keys=["canonical_dam_id","bulletin_date"]
check(11,"Page-1 observation-key uniqueness",not p1.duplicated(keys).any())
check(12,"Page-2 observation-key uniqueness",not p2n.duplicated(keys).any())
check(13,"Merged-key uniqueness",not merged.duplicated(keys).any())
union=len(pd.concat([p1[keys],p2n[keys]]).drop_duplicates()); check(14,"Outer-merge row reconciliation",len(merged)==union,(len(merged),union))
presence=merged.page_presence.value_counts().to_dict(); check(15,"Page-presence reconciliation",sum(presence.values())==len(merged) and set(presence)=={"both","left_only","right_only"},presence)
check(16,"Page-1 and Page-2 row preservation",presence.get("both",0)+presence.get("left_only",0)==len(p1) and presence.get("both",0)+presence.get("right_only",0)==len(p2))
check(17,"Null preservation",merged.isna().any().any() and p1.isna().any().any() and p2.isna().any().any())
f1=pd.read_csv(RUN_DIR/"page1"/"extraction_failures.csv"); f2=pd.read_csv(RUN_DIR/"page2"/"extraction_failures.csv")
check(18,"Failure table reconciliation",len(f1)==receipt["page1_summary"]["failure_count"] and len(f2)==receipt["page2_summary"]["failure_count"],(len(f1),len(f2)))
u1=pd.read_csv(RUN_DIR/"page1"/"unresolved_values.csv"); u2=pd.read_csv(RUN_DIR/"page2"/"unresolved_values.csv")
check(19,"Preserved-state union row reconciliation",len(u1)==receipt["page1_summary"]["unresolved_count"] and len(u2)==receipt["page2_summary"]["unresolved_count"],{"page1_preserved_rows":len(u1),"page2_reporting_union_rows":len(u2)})
c1=pd.read_csv(RUN_DIR/"page1"/"checksum_validation.csv.gz",low_memory=False); check(20,"Checksum rows never generate dam measurements","generates_dam_value" in c1 and not c1.generates_dam_value.astype(str).str.lower().isin(["true","1"]).any())

p2_summary=receipt["page2_summary"]
protected_rs=u2[u2["canonical_semantic"].eq("p2_rs_unresolved_prohibited")]
numeric_rs=protected_rs[protected_rs["parse_status"].eq("NUMERIC_PARSED_RS_UNRESOLVED_PROHIBITED")]
blank_rs=protected_rs[protected_rs["parse_status"].eq("SOURCE_BLANK")]
preserved_expressions=u2[u2["disposition"].eq("SOURCE_EXPRESSION_PRESERVED_NO_INFERENCE")]
check(21,"Page-2 extraction failures equal zero",p2_summary["failure_count"]==0,p2_summary["failure_count"])
check(22,"Page-2 actionable parser ambiguities equal zero",p2_summary["ambiguous_row_count"]==0,p2_summary["ambiguous_row_count"])
check(23,"Page-2 protected R.S. state count",len(protected_rs)==129472,len(protected_rs))
check(24,"Numeric protected R.S. count",len(numeric_rs)==81445,len(numeric_rs))
check(25,"Protected R.S. source-blank count",len(blank_rs)==48027,len(blank_rs))
check(26,"Preserved nonnumeric source-expression count",len(preserved_expressions)==16223,len(preserved_expressions))
check(27,"Page-2 scientific identity hash",p2_summary["scientific_identity_hash"]=="8b48836811dcfe80fe12db79295d360f1980e54ab4e51f69f559ab87ef55d361",p2_summary["scientific_identity_hash"])
assert len(u2)==len(protected_rs)+len(preserved_expressions)==145695
print("AUDIT_CHECKS_1_TO_27_COMPLETE")

AUDIT_CHECKS_1_TO_27_COMPLETE


In [4]:
db_path=RUN_DIR/"database"/"dam_data.db"
with sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro",uri=True) as conn:
    integrity=conn.execute("PRAGMA integrity_check").fetchone()[0]
    fk=conn.execute("PRAGMA foreign_key_check").fetchall()
    db_counts={t:conn.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0] for t in ("source_manifest","page1_observations","page2_observations","merged_daily")}
    db_merged=pd.read_sql_query("SELECT * FROM merged_daily ORDER BY bulletin_date, canonical_dam_id",conn)
check(28,"SQLite integrity",integrity=="ok",integrity)
check(29,"SQLite foreign keys",not fk,len(fk))
check(30,"Database table counts",db_counts=={"source_manifest":7964,"page1_observations":len(p1),"page2_observations":len(p2),"merged_daily":len(merged)},db_counts)
check(31,"CSV parseability",len(merged)>0 and len(compressed)>0)
check(32,"CSV column widths",len(merged.columns)==receipt["merged_column_count"]==66)
def canon(df): return df.sort_values(keys,kind="stable").reset_index(drop=True).fillna("<NULL>").astype(str)
check(33,"CSV-to-database equality",canon(merged).equals(canon(db_merged)))
check(34,"Plain CSV versus compressed CSV equality",canon(merged).equals(canon(compressed)))
check(35,"Final artifact hashes",sha256_file(RUN_DIR/"exports"/"merged_dam_daily.csv")==receipt["products"]["merged_csv"]["sha256"])
inv=list((RUN_DIR/"investigation").glob("*.csv"))
quality_path=RUN_DIR/"investigation"/"QUALITY_STATUS_SUMMARY.csv"
quality=pd.read_csv(quality_path) if quality_path.is_file() else pd.DataFrame()
quality_expected={
    "Page-2 extraction failures":0,
    "Page-2 actionable parser ambiguities":0,
    "Page-2 protected R.S. states":129472,
    "Numeric parsed but deliberately uninterpreted":81445,
    "Source blanks preserved as missing":48027,
    "Page-2 preserved nonnumeric source expressions":16223,
}
quality_actual=dict(zip(quality.get("status_category",[]),quality.get("rows",[])))
check(36,"Investigation CSV and quality-status reconciliation",len(inv)==14 and quality_actual==quality_expected,{"table_count":len(inv),"quality_status":quality_actual})
figs=list((RUN_DIR/"figures").glob("*.png"))
figure_names={p.name for p in figs}
check(37,"Investigation figure existence",len(figs)==12 and "08_quality_status.png" in figure_names and "08_failures_and_unresolved.png" not in figure_names and all(p.stat().st_size>0 for p in figs),sorted(figure_names))
print("AUDIT_CHECKS_28_TO_37_COMPLETE")

AUDIT_CHECKS_28_TO_37_COMPLETE


In [5]:
# Independent reconstruction of the governed one-to-one outer merge.
MERGE_KEYS=["canonical_dam_id","bulletin_date"]
p1_reference=p1.copy()
p2_reference=p2.rename(columns={"dam_id":"canonical_dam_id"}).copy()
assert not p1_reference.duplicated(MERGE_KEYS).any()
assert not p2_reference.duplicated(MERGE_KEYS).any()

p1_nonkeys=[c for c in p1_reference.columns if c not in MERGE_KEYS]
p2_nonkeys=[c for c in p2_reference.columns if c not in MERGE_KEYS]
expected_columns=(
    [c if c in MERGE_KEYS else f"page1__{c}" for c in p1_reference.columns]
    + [f"page2__{c}" for c in p2_reference.columns if c not in MERGE_KEYS]
    + ["page_presence"]
)
p1_prefixed=p1_reference.rename(columns={c:f"page1__{c}" for c in p1_nonkeys})
p2_prefixed=p2_reference.rename(columns={c:f"page2__{c}" for c in p2_nonkeys})
independent=p1_prefixed.merge(
    p2_prefixed,
    on=MERGE_KEYS,
    how="outer",
    validate="one_to_one",
    indicator="page_presence",
)
independent["page_presence"]=independent["page_presence"].astype(str)
independent=independent.sort_values(["bulletin_date","canonical_dam_id"],kind="stable").reset_index(drop=True)

def sorted_frame(frame):
    return frame.sort_values(MERGE_KEYS,kind="stable").reset_index(drop=True)

def exact_values_and_nulls(reference,candidate):
    reference=sorted_frame(reference)
    candidate=sorted_frame(candidate)
    if list(reference.columns)!=list(candidate.columns):
        return False
    if not reference.isna().equals(candidate.isna()):
        return False
    try:
        pd.testing.assert_frame_equal(reference,candidate,check_dtype=False,check_exact=True)
        return True
    except AssertionError:
        return False

p1_recovered=independent.loc[
    independent["page_presence"].isin(["both","left_only"]),
    [c if c in MERGE_KEYS else f"page1__{c}" for c in p1_reference.columns],
].rename(columns={f"page1__{c}":c for c in p1_nonkeys})
p2_recovered=independent.loc[
    independent["page_presence"].isin(["both","right_only"]),
    [c if c in MERGE_KEYS else f"page2__{c}" for c in p2_reference.columns],
].rename(columns={f"page2__{c}":c for c in p2_nonkeys})

p1_key_set=set(map(tuple,p1_reference[MERGE_KEYS].itertuples(index=False,name=None)))
p2_key_set=set(map(tuple,p2_reference[MERGE_KEYS].itertuples(index=False,name=None)))
expected_presence={
    key:("both" if key in p1_key_set and key in p2_key_set else "left_only" if key in p1_key_set else "right_only")
    for key in p1_key_set|p2_key_set
}
actual_presence=dict(zip(
    map(tuple,independent[MERGE_KEYS].itertuples(index=False,name=None)),
    independent["page_presence"],
))
merged_ordered=merged[expected_columns].sort_values(MERGE_KEYS,kind="stable").reset_index(drop=True)

check(38,"Independent merge exact prefixed column set",list(independent.columns)==expected_columns,{"expected_count":len(expected_columns),"actual_count":len(independent.columns),"missing":sorted(set(expected_columns)-set(independent.columns)),"extra":sorted(set(independent.columns)-set(expected_columns))})
check(39,"Independent merge preserves every Page-1 value and null",exact_values_and_nulls(p1_reference,p1_recovered),{"source_rows":len(p1_reference),"recovered_rows":len(p1_recovered)})
check(40,"Independent merge preserves every Page-2 value and null",exact_values_and_nulls(p2_reference,p2_recovered),{"source_rows":len(p2_reference),"recovered_rows":len(p2_recovered)})
check(41,"Independent page_presence correctness for every key",actual_presence==expected_presence,{"keys_checked":len(expected_presence)})
check(42,"Independent merge equals merged CSV field-by-field",exact_values_and_nulls(independent[expected_columns],merged_ordered),{"rows":len(independent),"columns":len(expected_columns)})
print("AUDIT_CHECKS_38_TO_42_COMPLETE")

AUDIT_CHECKS_38_TO_42_COMPLETE


In [6]:
after={str(p.relative_to(RUN_DIR)):sha256_file(p) for p in scientific_paths}
check(43,"Audit read-only behavior",before==after,{"before_count":len(before),"after_count":len(after),"changed":[k for k in before if before[k]!=after.get(k)]})
audit_results=pd.DataFrame(results)
AUDIT_DIR.mkdir(parents=True,exist_ok=True)
audit_results.to_csv(AUDIT_DIR/"audit_results.csv",index=False,lineterminator="\n")
summary={"run_id":receipt["run_id"],"pass_count":int((audit_results.status=="PASS").sum()),"warn_count":int((audit_results.status=="WARN").sum()),"fail_count":int((audit_results.status=="FAIL").sum()),"scientific_artifact_count":len(before),"scientific_hashes_unchanged":before==after}
(AUDIT_DIR/"audit_summary.json").write_text(json.dumps(summary,indent=2,sort_keys=True)+"\n",encoding="utf-8")
lines=["# Independent audit report","",f"- Run: {receipt['run_id']}",f"- PASS: {summary['pass_count']}",f"- WARN: {summary['warn_count']}",f"- FAIL: {summary['fail_count']}",f"- Scientific artifacts unchanged: {summary['scientific_hashes_unchanged']}","","| # | Check | Status | Details |","|---:|---|---|---|"]
for r in results: lines.append(f"| {r['check_number']} | {r['check_name']} | {r['status']} | {r['details'].replace('|','/')} |")
(AUDIT_DIR/"AUDIT_REPORT.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
display(audit_results)
print(json.dumps(summary,indent=2))
assert summary["fail_count"]==0 and summary["scientific_hashes_unchanged"]
print("AUDIT_COMPLETE = TRUE")

{
  "run_id": "RUN_20260831_FULL_01",
  "pass_count": 43,
  "warn_count": 0,
  "fail_count": 0,
  "scientific_artifact_count": 55,
  "scientific_hashes_unchanged": true
}
AUDIT_COMPLETE = TRUE


,check_number,check_name,status,details
0,1,Source root identity,PASS,{'page1': WindowsPath('/Downloa...
1,2,PDF counts,PASS,"{'page1': 3981, 'page2': 3983}"
2,3,Filename suffix purity,PASS,
3,4,Source manifest uniqueness,PASS,
4,5,Source hashes versus accepted inventory,PASS,
5,6,Page-1 engine version,PASS,page1-authoritative-0.2.3
6,7,Page-2 engine version,PASS,page2-authoritative-1.0.0
7,8,Frozen source hashes,PASS,
8,9,Page-1 source count,PASS,
9,10,Page-2 source count,PASS,


## Interpretation

A successful audit requires all 43 checks to pass and the before/after hashes of every scientific artifact to remain identical. WARN is reserved for a documented non-fatal limitation; any FAIL prevents declaring the pipeline complete.